# Book Recommender System (Medallion Data Lake)

Data Mining Lab Assignment – Book-Crossing dataset

Pipeline: **Bronze → Silver → Gold**, then item-based collaborative filtering on the Gold layer.

Dataset: Users, Books, Book-Ratings from the Book-Crossing dump.

## Step 1 – Setup + Bronze layer

Create the lake folders, download the raw CSVs, load them as-is (semicolon + latin-1), stamp an ingestion timestamp, and write Parquet. No cleaning here.

In [ ]:
# installs (Colab usually has pandas; pyarrow / sklearn may need a quick install)
!pip install -q pyarrow scikit-learn

In [ ]:
import os
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.neighbors import NearestNeighbors
import joblib
from IPython.display import display

warnings.filterwarnings("ignore")

# lake folders
for layer in ["bronze", "silver", "gold"]:
    os.makedirs(layer, exist_ok=True)

print("folders ready:", os.listdir("."))

In [ ]:
# download + unzip the Book-Crossing files
!wget -q -O book-crossings.zip https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
!unzip -o -q book-crossings.zip
!ls -lh BX-*.csv

In [ ]:
# raw load — semicolon separator, latin-1 encoding (files are not utf-8)
users_raw = pd.read_csv("BX-Users.csv", sep=";", encoding="latin-1", dtype=str)
books_raw = pd.read_csv("BX-Books.csv", sep=";", encoding="latin-1", dtype=str, on_bad_lines="skip")
ratings_raw = pd.read_csv("BX-Book-Ratings.csv", sep=";", encoding="latin-1", dtype=str)

print("users:", users_raw.shape)
print("books:", books_raw.shape)
print("ratings:", ratings_raw.shape)

users_raw.head(3)

In [ ]:
# bronze = untouched source + ingestion timestamp
ingestion_ts = datetime.now(timezone.utc)

users_raw["ingestion_timestamp"] = ingestion_ts
books_raw["ingestion_timestamp"] = ingestion_ts
ratings_raw["ingestion_timestamp"] = ingestion_ts

users_raw.to_parquet("bronze/users.parquet", index=False)
books_raw.to_parquet("bronze/books.parquet", index=False)
ratings_raw.to_parquet("bronze/ratings.parquet", index=False)

print("wrote bronze parquet files")
print(os.listdir("bronze"))

## Step 2 – EDA on Bronze

Reading the bronze tables back and poking around. Notes under each check say what I'll fix in Silver.

In [ ]:
users = pd.read_parquet("bronze/users.parquet")
books = pd.read_parquet("bronze/books.parquet")
ratings = pd.read_parquet("bronze/ratings.parquet")

print("=== shape ===")
print("users", users.shape, "| books", books.shape, "| ratings", ratings.shape)

print("\n=== dtypes ===")
print("users:\n", users.dtypes)
print("books:\n", books.dtypes)
print("ratings:\n", ratings.dtypes)

print("\n=== missing values ===")
print("users:\n", users.isna().sum())
print("books:\n", books.isna().sum())
print("ratings:\n", ratings.isna().sum())

**Findings (shape / types / missing):** Users ~279k × 4, books ~271k × 9, ratings ~1.15M × 4. Everything came in as strings because we forced `dtype=str` on read (fine for bronze). Big hole in `Age` (~110k nulls). Books are mostly complete; a couple authors/publishers missing. Ratings look complete. → In Silver I'll cast numerics properly and impute age.

In [ ]:
# ratings are still strings here
ratings["Book-Rating"] = pd.to_numeric(ratings["Book-Rating"], errors="coerce")

print(ratings["Book-Rating"].value_counts().sort_index())
print("\nzeros:", (ratings["Book-Rating"] == 0).sum())
print("explicit (1-10):", ((ratings["Book-Rating"] >= 1) & (ratings["Book-Rating"] <= 10)).sum())

ratings["Book-Rating"].hist(bins=11)

**Findings (ratings):** Most rows are 0 (~716k). In this dataset 0 means "implicit / no explicit rating", not a real score of zero. Explicit ratings cluster toward the high end (7–10). → Drop all 0s in Silver and keep 1–10 only.

In [ ]:
users["Age"] = pd.to_numeric(users["Age"], errors="coerce")
books["Year-Of-Publication"] = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")

print("Age describe:\n", users["Age"].describe())
print("ages <5 or >100:", ((users["Age"] < 5) | (users["Age"] > 100)).sum())

print("\nYear describe:\n", books["Year-Of-Publication"].describe())
print("year == 0:", (books["Year-Of-Publication"] == 0).sum())
print("year > current:", (books["Year-Of-Publication"] > datetime.now().year).sum())

users["Age"].clip(upper=120).hist(bins=40)

**Findings (age / year):** Ages go from 0 up to 244 — clearly junk. ~1.2k ages outside 5–100. Pub years include 0 (~4.6k) and a handful of future years (up to 2050). → Null out impossible ages then fill with median; null out year 0 / future years.

In [ ]:
rpu = ratings.groupby("User-ID").size()
rpb = ratings.groupby("ISBN").size()

print("ratings / user:\n", rpu.describe())
print("\nratings / book:\n", rpb.describe())
print("\n% users with <=2 ratings:", (rpu <= 2).mean() * 100)
print("% books with <=2 ratings:", (rpb <= 2).mean() * 100)

**Findings (activity):** Median user has 1 rating, median book also 1. Long tail is huge — a few power users / popular books, everyone else barely shows up. → Need a minimum-ratings filter or the similarity matrix will be garbage.

In [ ]:
n_users = ratings["User-ID"].nunique()
n_books = ratings["ISBN"].nunique()
n_pairs = len(ratings)
possible = n_users * n_books
sparsity = 1 - (n_pairs / possible)

print(f"users={n_users}, books={n_books}, ratings={n_pairs}")
print(f"possible pairs={possible:,}")
print(f"sparsity={sparsity*100:.4f}%  (only {n_pairs/possible*100:.5f}% filled)")

**Findings (sparsity):** ~99.997% empty. Dense matrix is not an option — must keep this sparse in Gold.

## Step 3 – Silver layer (cleaning)

Cleaning rules from the brief, plus a min-ratings cutoff justified by the EDA above.

In [ ]:
# start fresh from bronze so this cell is re-runnable
users = pd.read_parquet("bronze/users.parquet").drop_duplicates()
books = pd.read_parquet("bronze/books.parquet").drop_duplicates()
ratings = pd.read_parquet("bronze/ratings.parquet").drop_duplicates()

print("after drop_duplicates:", users.shape, books.shape, ratings.shape)

# ratings: keep explicit 1-10 only
ratings["Book-Rating"] = pd.to_numeric(ratings["Book-Rating"], errors="coerce")
ratings = ratings[(ratings["Book-Rating"] >= 1) & (ratings["Book-Rating"] <= 10)].copy()
print("after dropping 0-ratings:", ratings.shape)

# ages: impossible -> null, then median fill
users["Age"] = pd.to_numeric(users["Age"], errors="coerce")
users.loc[(users["Age"] < 5) | (users["Age"] > 100), "Age"] = np.nan
median_age = users["Age"].median()
users["Age"] = users["Age"].fillna(median_age)
print("median age used for fill:", median_age)

# split Location -> city, state, country
parts = users["Location"].fillna("").str.split(",", n=2, expand=True)
users["city"] = parts[0].str.strip().str.title() if 0 in parts.columns else ""
users["state"] = parts[1].str.strip().str.title() if 1 in parts.columns else ""
users["country"] = parts[2].str.strip().str.title() if 2 in parts.columns else ""
users = users.drop(columns=["Location"])

# publication year cleanup
books["Year-Of-Publication"] = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
yr = datetime.now().year
books.loc[(books["Year-Of-Publication"] == 0) | (books["Year-Of-Publication"] > yr), "Year-Of-Publication"] = np.nan

# text cleanup + drop image urls
for col in ["Book-Title", "Book-Author", "Publisher"]:
    books[col] = books[col].astype(str).str.strip().str.title()
    books[col] = books[col].replace({"Nan": np.nan, "None": np.nan})

img_cols = [c for c in books.columns if c.startswith("Image-URL")]
books = books.drop(columns=img_cols)

# ids as clean strings
users["User-ID"] = users["User-ID"].astype(str).str.strip()
books["ISBN"] = books["ISBN"].astype(str).str.strip()
ratings["User-ID"] = ratings["User-ID"].astype(str).str.strip()
ratings["ISBN"] = ratings["ISBN"].astype(str).str.strip()

# only ratings that match known users + books
ratings = ratings[
    ratings["User-ID"].isin(set(users["User-ID"]))
    & ratings["ISBN"].isin(set(books["ISBN"]))
].copy()
print("after referential filter:", ratings.shape)

In [ ]:
# min activity filter
# EDA showed median = 1 rating for both users and books, so cosine on that is noise.
# Tried 5 / 10 / 20 on explicit ratings:
#   5  -> ~152k ratings (still pretty sparse users)
#   10 -> ~90k ratings, ~6.8k users, ~5.6k books  <-- picking this
#   20 -> ~47k ratings (a bit aggressive, lose a lot of signal)
# Threshold = 10: enough overlap for item similarity without exploding runtime in Colab.

MIN_RATINGS = 10

user_n = ratings.groupby("User-ID").size()
book_n = ratings.groupby("ISBN").size()
keep_users = user_n[user_n >= MIN_RATINGS].index
keep_books = book_n[book_n >= MIN_RATINGS].index

ratings = ratings[ratings["User-ID"].isin(keep_users) & ratings["ISBN"].isin(keep_books)].copy()
users = users[users["User-ID"].isin(ratings["User-ID"].unique())].copy()
books = books[books["ISBN"].isin(ratings["ISBN"].unique())].copy()

print(f"MIN_RATINGS={MIN_RATINGS}")
print("silver users:", users.shape)
print("silver books:", books.shape)
print("silver ratings:", ratings.shape)

users.to_parquet("silver/users.parquet", index=False)
books.to_parquet("silver/books.parquet", index=False)
ratings.to_parquet("silver/ratings.parquet", index=False)
print("saved silver/")

## Step 4 – Gold layer

Model-ready stuff: sparse user–book matrix (books × users), a small book lookup, and some aggregates.

In [ ]:
users = pd.read_parquet("silver/users.parquet")
books = pd.read_parquet("silver/books.parquet")
ratings = pd.read_parquet("silver/ratings.parquet")

user_ids = sorted(ratings["User-ID"].unique())
book_ids = sorted(ratings["ISBN"].unique())

user_index = {u: i for i, u in enumerate(user_ids)}
book_index = {b: i for i, b in enumerate(book_ids)}

row = ratings["ISBN"].map(book_index).to_numpy()
col = ratings["User-ID"].map(user_index).to_numpy()
val = ratings["Book-Rating"].astype(float).to_numpy()

# rows = books, cols = users  (needed for item-item CF)
rating_matrix = csr_matrix((val, (row, col)), shape=(len(book_ids), len(user_ids)))
print("rating matrix:", rating_matrix.shape, "nnz:", rating_matrix.nnz)
print("sparsity:", 1 - rating_matrix.nnz / (rating_matrix.shape[0] * rating_matrix.shape[1]))

book_lookup = books[["ISBN", "Book-Title", "Book-Author"]].drop_duplicates("ISBN")

user_aggs = (
    ratings.groupby("User-ID")
    .agg(avg_rating=("Book-Rating", "mean"), n_ratings=("Book-Rating", "count"))
    .reset_index()
)
book_aggs = (
    ratings.groupby("ISBN")
    .agg(avg_rating=("Book-Rating", "mean"), n_ratings=("Book-Rating", "count"))
    .reset_index()
)

save_npz("gold/rating_matrix.npz", rating_matrix)
joblib.dump(
    {
        "user_ids": user_ids,
        "book_ids": book_ids,
        "user_index": user_index,
        "book_index": book_index,
    },
    "gold/matrix_index.joblib",
)
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)
user_aggs.to_parquet("gold/user_aggregates.parquet", index=False)
book_aggs.to_parquet("gold/book_aggregates.parquet", index=False)

print("gold files:", os.listdir("gold"))
book_lookup.head()

## Step 5 – Item-based collaborative filtering

Cosine nearest neighbors over the book vectors. Then `similar_books(title)` and `recommend(user_id)`.

In [ ]:
# load gold
rating_matrix = load_npz("gold/rating_matrix.npz")
meta = joblib.load("gold/matrix_index.joblib")
book_lookup = pd.read_parquet("gold/book_lookup.parquet")
ratings = pd.read_parquet("silver/ratings.parquet")

user_ids = meta["user_ids"]
book_ids = meta["book_ids"]
user_index = meta["user_index"]
book_index = meta["book_index"]

isbn_to_title = dict(zip(book_lookup["ISBN"], book_lookup["Book-Title"]))
isbn_to_author = dict(zip(book_lookup["ISBN"], book_lookup["Book-Author"]))

# title lookup (lowercased). if duplicates, first one wins
title_to_isbn = {}
for _, r in book_lookup.iterrows():
    title_to_isbn.setdefault(str(r["Book-Title"]).lower(), r["ISBN"])

# fit item-item model (each row is a book)
nn_model = NearestNeighbors(metric="cosine", algorithm="brute")
nn_model.fit(rating_matrix)
print("NearestNeighbors fitted on", rating_matrix.shape)

In [ ]:
def _resolve_isbn(title: str):
    key = title.lower().strip()
    if key in title_to_isbn:
        return title_to_isbn[key]
    # soft match: substring
    hits = [t for t in title_to_isbn if key in t]
    if hits:
        return title_to_isbn[hits[0]]
    return None


def similar_books(title, n=10):
    """Return top-n books most similar to the given title."""
    isbn = _resolve_isbn(title)
    if isbn is None:
        return f"couldn't find a book matching '{title}'"
    if isbn not in book_index:
        return f"'{title}' is not in the gold matrix (filtered out by min ratings)"

    idx = book_index[isbn]
    dists, idxs = nn_model.kneighbors(rating_matrix[idx], n_neighbors=n + 1)

    results = []
    for dist, i in zip(dists.flatten()[1:], idxs.flatten()[1:]):  # skip self
        bid = book_ids[i]
        results.append(
            {
                "title": isbn_to_title.get(bid, bid),
                "author": isbn_to_author.get(bid, ""),
                "similarity": float(1 - dist),
            }
        )
    return pd.DataFrame(results)


def recommend(user_id, n=10):
    """Recommend n books for a user using item-item CF.

    Looks at books they rated highly (>=7), finds similar books,
    scores by similarity * user rating, drops already-read titles.
    """
    user_id = str(user_id)
    if user_id not in user_index:
        return f"user {user_id} not in gold set (maybe < {10} ratings)"

    u_rats = ratings[ratings["User-ID"] == user_id]
    liked = u_rats[u_rats["Book-Rating"] >= 7].sort_values("Book-Rating", ascending=False)
    if liked.empty:
        liked = u_rats.sort_values("Book-Rating", ascending=False).head(5)

    already = set(u_rats["ISBN"])
    scores = {}

    for _, row in liked.iterrows():
        isbn = row["ISBN"]
        rating = float(row["Book-Rating"])
        if isbn not in book_index:
            continue
        idx = book_index[isbn]
        dists, idxs = nn_model.kneighbors(rating_matrix[idx], n_neighbors=20)
        for dist, i in zip(dists.flatten()[1:], idxs.flatten()[1:]):
            cand = book_ids[i]
            if cand in already:
                continue
            sim = 1.0 - float(dist)
            scores[cand] = scores.get(cand, 0.0) + sim * rating

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]
    rows = []
    for isbn, score in ranked:
        rows.append(
            {
                "title": isbn_to_title.get(isbn, isbn),
                "author": isbn_to_author.get(isbn, ""),
                "score": score,
            }
        )
    return pd.DataFrame(rows)


# quick sanity: similar books for a known title
sample_title = book_lookup["Book-Title"].iloc[0]
print("example title:", sample_title)
similar_books(sample_title, 5)

In [ ]:
# try a well-known book if it's in the filtered set
hp = [t for t in title_to_isbn if "harry potter" in t and "sorcerer" in t]
print("hp matches:", hp[:5])
if hp:
    display(similar_books(hp[0], 10))

## Step 6 – Demo / sanity check

Pick three active users, show what they liked, then what `recommend()` returns.

In [ ]:
def show_user_demo(user_id, n=10):
    user_id = str(user_id)
    u_rats = ratings[ratings["User-ID"] == user_id].copy()
    u_rats["title"] = u_rats["ISBN"].map(isbn_to_title)
    liked = u_rats.sort_values("Book-Rating", ascending=False).head(8)

    print("=" * 70)
    print(f"USER {user_id}  ({len(u_rats)} ratings in gold set)")
    print("- liked / high ratings -")
    display(liked[["title", "Book-Rating"]])
    print("- recommendations -")
    display(recommend(user_id, n=n))


# three users with plenty of ratings so the demo is readable
top_users = (
    ratings.groupby("User-ID").size().sort_values(ascending=False).head(15).index.tolist()
)
demo_users = top_users[:3]
print("demo users:", demo_users)

for uid in demo_users:
    show_user_demo(uid, n=10)

In [ ]:
# one more check with a mid-activity user (not just the top power users)
mid_users = (
    ratings.groupby("User-ID").size().sort_values(ascending=False).iloc[40:60].index.tolist()
)
show_user_demo(mid_users[0], n=10)

### Notes / wrap-up

- Bronze kept the raw dump (latin-1, semicolons, zeros, weird ages, image URLs, etc.).
- Silver dropped implicit 0-ratings, fixed ages/years, split location, cleaned text, enforced FK-ish filters, and kept users/books with ≥10 ratings.
- Gold stores a sparse books×users matrix plus lookup/aggregates.
- Recommender is plain item-item CF with cosine similarity. Similar-books for Harry Potter pulls other HP books, which is a decent smoke test. Recommendations for heavy raters are a mix of popular overlap titles — expected with this sparsity.

Threshold choice (10) was the main modelling decision; lower gets noisier neighbors, higher shrinks the catalog too hard for a Colab demo.